# Chapter 3: Feature Engineering

## Your Model Can't Read English — Here's How to Translate

At this point in the pipeline, we have validated data with standardized column names. But we're not ready for a model yet.

A model is just math. It multiplies, adds, and compares numbers. It cannot:
- Understand that `contract_type = "month-to-month"` means high risk
- Handle a blank cell (NaN) — it crashes or produces garbage
- Fairly compare `tenure_months` (range: 1-72) with `monthly_charges` (range: 18-118)

Feature engineering is the **prep chef** — it takes raw, validated ingredients and chops them into a form the model can digest. Four steps:

| Step | What It Does | Why |
|------|-------------|-----|
| Imputation | Fills blank cells | Model crashes on NaN |
| Encoding | Converts categories to numbers | Model only speaks numbers |
| Interaction features | Creates meaningful combinations | Captures relationships |
| Scaling | Puts everything on same range [0,1] | Prevents unfair dominance |

In [1]:
import pandas as pd
import numpy as np
from churn_pipeline.steps.feature_engineering import engineer_features, FeatureArtifacts

## Step 0: Start With Validated Data

This is what comes out of Chapter 2 — data that passed the bouncer. Column names are standardized, types are (mostly) correct, but there are still blanks, strings, and uneven scales.

In [2]:
# Realistic validated data — notice the problems a model can't handle
validated_data = pd.DataFrame({
    "customer_id": ["C001", "C002", "C003", "C004", "C005", "C006", "C007"],
    "tenure_months": [1, 34, 2, 45, None, 12, 60],        # NaN! Model will crash
    "monthly_charges": [29.85, 56.95, 53.85, 42.30, 89.10, None, 105.45],  # NaN!
    "total_charges": [29.85, 1889.5, 108.15, 1840.75, 2000.0, 500.0, 6327.0],
    "churn_label": [0, 1, 1, 0, 1, 0, 0],
    "contract_type": ["month-to-month", "one_year", "month-to-month", "two_year",
                       "month-to-month", None, "two_year"],  # String! And NaN!
    "support_tickets": [0, 3, 5, 0, 2, 1, 0],
    "payment_method": ["electronic_check", "credit_card", "electronic_check",
                        "bank_transfer", "credit_card", "electronic_check", "bank_transfer"],
})

print("Validated data (still has problems for a model):")
print(validated_data)
print(f"\nNull values per column:")
print(validated_data.isnull().sum())
print(f"\nColumn types:")
print(validated_data.dtypes)

Validated data (still has problems for a model):
  customer_id  tenure_months  monthly_charges  total_charges  churn_label  \
0        C001            1.0            29.85          29.85            0   
1        C002           34.0            56.95        1889.50            1   
2        C003            2.0            53.85         108.15            1   
3        C004           45.0            42.30        1840.75            0   
4        C005            NaN            89.10        2000.00            1   
5        C006           12.0              NaN         500.00            0   
6        C007           60.0           105.45        6327.00            0   

    contract_type  support_tickets    payment_method  
0  month-to-month                0  electronic_check  
1        one_year                3       credit_card  
2  month-to-month                5  electronic_check  
3        two_year                0     bank_transfer  
4  month-to-month                2       credit_card  
5   

## The Transformation: Before and After

One function call takes us from messy validated data to a clean numeric matrix. Let's see the full transformation.

In [3]:
# Training mode: learn the transformation rules AND apply them
feature_matrix, artifacts = engineer_features(validated_data, fit=True)

print("AFTER feature engineering:")
print(f"  Shape: {feature_matrix.shape} (7 customers × {feature_matrix.shape[1]} features)")
print(f"  Data type: {feature_matrix.dtype}")
print(f"  Any NaN values: {np.isnan(feature_matrix).any()}")
print(f"  Value range: [{feature_matrix.min():.3f}, {feature_matrix.max():.3f}]")
print(f"\nFeature names (in order):")
for i, name in enumerate(artifacts.feature_names):
    print(f"  {i}: {name}")

AFTER feature engineering:
  Shape: (7, 7) (7 customers × 7 features)
  Data type: float64
  Any NaN values: False
  Value range: [0.000, 1.000]

Feature names (in order):
  0: tenure_months
  1: monthly_charges
  2: total_charges
  3: support_tickets
  4: contract_type
  5: payment_method
  6: interaction_charges_x_tenure


In [4]:
# Let's look at the actual numbers
feature_df = pd.DataFrame(feature_matrix, columns=artifacts.feature_names)
print("The feature matrix (what the model actually sees):")
print(feature_df.round(3))

The feature matrix (what the model actually sees):
   tenure_months  monthly_charges  total_charges  support_tickets  \
0          0.000            0.000          0.000              0.0   
1          0.559            0.358          0.295              0.6   
2          0.017            0.317          0.012              1.0   
3          0.746            0.165          0.288              0.0   
4          0.373            0.784          0.313              0.4   
5          0.186            0.338          0.075              0.2   
6          1.000            1.000          1.000              0.0   

   contract_type  payment_method  interaction_charges_x_tenure  
0            0.0             1.0                         0.000  
1            0.5             0.5                         0.303  
2            0.0             1.0                         0.012  
3            1.0             0.0                         0.298  
4            0.0             0.5                         0.321  
5     

## Unpacking Each Step

### Step 1: Imputation — Filling the Blanks

Missing values are poison for models. A blank cell will either crash the math or produce garbage output. So we need to fill every blank with *something*.

But what do we fill it with? There are several strategies. Here's what each one actually does:

**Median** — Sort all existing values from lowest to highest, pick the one in the exact middle. If you have 100 customers with tenure values, the median is the 50th value when sorted. Unlike the average, one crazy outlier (tenure = 999) doesn't move the median much. This is what we use for numbers.

**Mean (average)** — Add up all values and divide by how many there are. Simple, but if one customer has $10,000 monthly charges (data entry error) and 99 others have $50-$100, the mean jumps to ~$150. The median would still be ~$75. Mean is only safe when you're confident there are no outliers.

**Mode** — The most common value. If 60% of customers have \"month-to-month\" contracts, the mode is \"month-to-month\". We use this for categories (text values). The risk: if you have many blanks, they ALL become the most popular category, which may inflate that group.

**Forward/backward fill** — Copy the value from the row above (or below). This ONLY makes sense when rows are sorted by time (like daily stock prices — today's price is probably close to yesterday's). For a shuffled list of customers, the row above is a random person — copying their value makes no sense.

**Group-based** — \"Fill this customer's missing monthly_charges with the median of customers who have the same contract type.\" Month-to-month customers might average $70/month while two-year customers average $40/month — so we fill based on which group they belong to. Smarter than global median, but needs enough data in each group.

**Model-based (nearest neighbors)** — Find the 5 customers most similar to this one (based on their OTHER columns that aren't missing) and average their values. If a customer has tenure=3, charges=$90, and missing total_charges, find 5 other customers with similar tenure and charges, and use their average total_charges. Smart but slow, and can overfit if you're not careful.

**Constant + flag** — Fill with 0 (or -1), AND create a new column called `tenure_months_was_missing` (1 if it was blank, 0 if it was real). This lets the model learn \"customers with missing tenure behave differently\" — which can be true! Maybe new customers have blank fields, and being new IS a churn signal.

**Drop the row** — Just delete customers with missing values. Simple and honest (no made-up data), but you lose information. Only reasonable if less than 1% of rows are affected.

**What we chose and why:**
- **Numeric → median.** Robust to outliers. Safe default for messy data.
- **Categorical → mode.** Least-wrong single guess for blanks in text columns.

**In a more sophisticated pipeline, you might:**
- Use group-based imputation: fill missing `monthly_charges` with the median of customers who have the same `contract_type`
- Add a `tenure_months_was_missing` binary flag — sometimes the fact that data is missing IS a signal (e.g., very new customers might have missing fields)
- Use nearest-neighbor imputation: find the 5 most similar customers and average their values

For this pipeline, simple median/mode works well because our datasets are large enough that a few imputed values don't significantly change the model.

In [5]:
# What imputation values were learned?
print("Learned imputation values (what fills the blanks):")
print("=" * 50)
for field, value in artifacts.impute_values.items():
    print(f"  {field:20s} → {value}")

print("\nFor example:")
print(f"  tenure_months had a NaN → filled with median = {artifacts.impute_values.get('tenure_months')}")
print(f"  monthly_charges had a NaN → filled with median = {artifacts.impute_values.get('monthly_charges')}")
print(f"  contract_type had a NaN → filled with mode = '{artifacts.impute_values.get('contract_type')}'")

Learned imputation values (what fills the blanks):
  tenure_months        → 23.0
  monthly_charges      → 55.400000000000006
  total_charges        → 1840.75
  support_tickets      → 1.0
  contract_type        → month-to-month
  payment_method       → electronic_check

For example:
  tenure_months had a NaN → filled with median = 23.0
  monthly_charges had a NaN → filled with median = 55.400000000000006
  contract_type had a NaN → filled with mode = 'month-to-month'


### Step 2: Encoding — Categories Become Numbers

A model can't multiply by `"month-to-month"`. It only understands numbers. So we need to convert every category into a number. But HOW you do this matters a lot.

#### The Options

There are several ways to turn categories into numbers. Each works differently:

**Label Encoding (what we use)** — Give each category a sequential number: month-to-month=0, one_year=1, two_year=2. Simple, compact (one column stays one column). The numbers don't mean \"month-to-month is LESS than one_year\" — they're just IDs. Tree-based models (like XGBoost) understand this because they split on \"is the value ≤ 1?\" and can treat each number independently.

**One-Hot Encoding** — Instead of one column with numbers 0/1/2, create THREE separate yes/no columns: `is_monthly` (1 or 0), `is_one_year` (1 or 0), `is_two_year` (1 or 0). Each customer has exactly one of these set to 1. This is safer for linear models because it doesn't imply any ordering. Downside: if you have 50 categories, you get 50 new columns.

**Target Encoding** — Replace each category with the average outcome (churn rate) of customers in that group. Month-to-month customers churn at 43%, so `month-to-month → 0.43`. One-year customers churn at 11%, so `one_year → 0.11`. This directly encodes \"how risky is this category?\" Danger: it uses the answer (churn labels) to create the input — which can trick the model into memorizing the training data instead of learning general patterns.

**Frequency Encoding** — Replace each category with how often it appears in the dataset. If 55% of customers are month-to-month, that category becomes `0.55`. This tells the model \"how common is this type?\" — rare categories might behave differently from common ones. Downside: two different categories with the same frequency become indistinguishable.

**Ordinal Encoding** — Like label encoding, but the numbers have REAL meaning because the categories have a natural order. Example: satisfaction ratings low=1, medium=2, high=3. Here, 3 really IS greater than 1 in a meaningful way. Only use this when the order genuinely exists in the real world.

#### Why We Chose Label Encoding

We use **label encoding** (each category = one integer) because:

1. **XGBoost handles it well.** Tree-based models split on "is value ≤ 1?" — they naturally handle arbitrary integer encodings without assuming order. A tree can learn that 0 (month-to-month) is high risk and 2 (two_year) is low risk, regardless of the numbers.

2. **Keeps column count small.** With one-hot encoding, 3 contract types → 3 columns. With 10 payment methods → 10 columns. Label encoding keeps it to 1 column each.

3. **Simple to store and reproduce.** The encoder just remembers "month-to-month=0, one_year=1, two_year=2" — easy to save and apply during scoring.

**When would label encoding be WRONG?**
- For linear/logistic regression: it would assume month-to-month(0) < one_year(1) < two_year(2), treating them as ordered. Use one-hot encoding instead.
- For categories with 100+ unique values: encoding creates numbers 0-99 that have no meaning. Target encoding or embedding layers work better.

#### What Happens with Unknown Categories During Scoring?

New data might contain a category the model never saw during training (e.g., a new payment method `"paypal"`). Our encoder handles this by substituting the first known category — not ideal, but the model won't crash. A more robust approach would add an explicit "unknown" category during training.

The key rule: **whatever encoding you use during training, you MUST use the exact same during scoring.** If `one_year=1` during training, it must be `1` during scoring too.

In [6]:
# What encodings were learned?
print("Learned category encodings:")
print("=" * 50)
for col, encoder in artifacts.encoders.items():
    print(f"\n  {col}:")
    for i, cls in enumerate(encoder.classes_):
        print(f"    '{cls}' → {i}")

Learned category encodings:

  contract_type:
    'month-to-month' → 0
    'one_year' → 1
    'two_year' → 2

  payment_method:
    'bank_transfer' → 0
    'credit_card' → 1
    'electronic_check' → 2


### Step 3: Interaction Features — Combinations That Tell a Story

Some relationships are invisible when looking at features alone:

- Customer A: monthly_charges=100, tenure=2 months → paid $200 total
- Customer B: monthly_charges=100, tenure=36 months → paid $3,600 total

Same monthly charges. Completely different relationships with the company. The interaction feature `monthly_charges × tenure_months` captures "approximate lifetime value" — something neither feature alone conveys.

#### Other Feature Engineering Techniques (Not Used Here, But Common)

There are many other ways to create useful features. Here's what each one actually DOES, explained before named:

**Ratios — Divide one number by another to get a more meaningful number**

If a customer paid $3,600 total over 36 months, what they pay per month is $3600/36 = $100. The ratio `total_charges / tenure_months` tells you their average monthly cost — which might be more informative than either number alone. A customer who paid $3,600 over 3 months ($1,200/mo) is very different from one who paid $3,600 over 36 months ($100/mo).

**Binning — Turn a continuous number into a handful of labeled groups**

Instead of saying "tenure = 4 months", say "tenure = new customer (0-6 months)". Instead of the exact number, you put people into buckets: new (0-6), growing (7-24), established (25+). Why? Because sometimes the exact number doesn't matter — what matters is which SIDE of a threshold you're on. A 5-month customer and a 6-month customer behave similarly. But a 6-month and a 25-month customer behave very differently.

**Log transform — Squishing a very spread-out range into something manageable**

Imagine a column where most values are between $20-$200, but a few are $5,000 or $50,000. Those extreme values dominate everything. Taking the logarithm (log) compresses the scale: log($20)=1.3, log($200)=2.3, log($5000)=3.7, log($50000)=4.7. Now the extremes aren't 250× bigger — they're only 3.6× bigger. This helps models that are sensitive to scale (like linear regression). Think of it as switching from a ruler measured in millimeters to one measured in "powers of 10."

**Polynomial features — Giving a straight-line model the ability to learn curves**

A linear model can only learn straight-line relationships: "more tenure = less churn, at a constant rate." But reality is curved: going from 1 month to 6 months reduces risk a LOT, going from 50 to 55 months barely changes it. If you feed the model `tenure²` (tenure times itself), it can learn this curve. 1²=1, 6²=36, 50²=2500, 55²=3025 — the early months are amplified, the later months less so. This is only needed for linear models. XGBoost learns curves naturally because trees can split at any point.

**Date features — Extracting useful information from timestamps**

If you have a signup date like `2023-03-15`, the raw date is useless to a model. But you can extract: month (March = Q1 signup), day of week (Wednesday), days since signup (how long ago), or season. Maybe customers who sign up in January (New Year's resolution) churn faster than those who sign up in June.

**Aggregations — "How does this customer compare to their group?"**

Instead of just "this customer filed 5 support tickets", compute "the average for their contract type is 2 tickets — so they filed 2.5× more than their peers." Group statistics add context: 5 tickets might be normal for month-to-month customers but alarming for two-year contract holders.

**Text features (TF-IDF) — Turning words into numbers**

If you have free-text data (like support ticket descriptions: "I can't connect to the internet AGAIN"), you can't feed words directly to a model. TF-IDF measures how important each word is: common words like "the" get low scores, distinctive words like "cancel" or "frustrated" get high scores. The result is a column of numbers — one per important word — that captures what the text is about. We don't have free text in this dataset, but if you did (say, customer complaint notes), this is how you'd use it.

**What we chose and why:**
- Just one interaction: `monthly_charges × tenure_months`. This captures lifetime value — the single most predictive derived feature for churn.
- We keep it simple because XGBoost already handles non-linear relationships internally (it can learn "if tenure < 6 AND charges > 80 → high risk" without us creating that interaction manually).
- For a linear model, you'd need many more engineered features because the model can't discover interactions on its own.

In [7]:
# Show the interaction feature alongside its components
print("Interaction feature: monthly_charges × tenure_months")
print("=" * 55)
print(f"{'Customer':<10} {'Charges':<10} {'Tenure':<10} {'Interaction (raw)'}")
print("-" * 55)

for i in range(len(validated_data)):
    charges = validated_data['monthly_charges'].iloc[i]
    tenure = validated_data['tenure_months'].iloc[i]
    # After imputation, the interaction would be computed from filled values
    charges_filled = charges if not pd.isna(charges) else artifacts.impute_values.get('monthly_charges', 0)
    tenure_filled = tenure if not pd.isna(tenure) else artifacts.impute_values.get('tenure_months', 0)
    interaction = charges_filled * tenure_filled
    print(f"  C{i+1:<7} ${charges_filled:<9.2f} {tenure_filled:<10.0f} {interaction:.2f}")

Interaction feature: monthly_charges × tenure_months
Customer   Charges    Tenure     Interaction (raw)
-------------------------------------------------------
  C1       $29.85     1          29.85
  C2       $56.95     34         1936.30
  C3       $53.85     2          107.70
  C4       $42.30     45         1903.50
  C5       $89.10     23         2049.30
  C6       $55.40     12         664.80
  C7       $105.45    60         6327.00


### Step 4: Scaling — Leveling the Playing Field

After imputation and encoding, we have all numbers — but they're on wildly different scales:
- `tenure_months`: 1 to 72
- `monthly_charges`: 18 to 118
- `total_charges`: 29 to 8,000+
- `support_tickets`: 0 to 10

#### Why Scale?

Some models (linear regression, neural networks, KNN) treat bigger numbers as more important. If `total_charges` ranges 0-8000 and `support_tickets` ranges 0-10, the model might ignore tickets simply because the numbers are smaller. Scaling puts everything on equal footing.

**For XGBoost specifically, scaling is technically not required** — trees split on "is value ≤ X?" regardless of scale. But we do it anyway because:
1. It makes SHAP values more comparable across features
2. It keeps the feature matrix numerically stable
3. It makes the pipeline portable to other algorithms later

#### Scaling Options

| Method | Formula | Range | When to Use |
|--------|---------|-------|-------------|
| **Min-Max** (what we use) | (x - min) / (max - min) | [0, 1] | When you want bounded values, no assumptions about distribution |
| **Standard (Z-score)** | (x - mean) / std | unbounded (centered at 0) | When data is roughly bell-shaped, common for linear models |
| **Robust** | (x - median) / IQR | unbounded | When data has many outliers (uses median instead of mean) |
| **No scaling** | keep original values | original | Tree-based models that don't need it |

**What we chose and why:**
Min-Max scaling squishes everything to [0, 1]:

```
scaled_value = (value - min) / (max - min)
```

Now 0.5 means "halfway between lowest and highest" regardless of the original unit. A tenure of 36 months (out of 72 max) and a charge of $68 (out of $118 max) both become ~0.5.

**Important:** We clip values to [0, 1] during scoring. If new data has tenure=84 (beyond the training max of 72), it gets clipped to 1.0 instead of going to 1.17. This prevents out-of-range values that the model never saw during training.

In [8]:
# Show the scaling ranges learned
print("Scaling ranges (learned from training data):")
print("=" * 60)
print(f"{'Feature':<30} {'Min (→0.0)':<15} {'Max (→1.0)'}")
print("-" * 60)
for i, name in enumerate(artifacts.feature_names):
    min_val = artifacts.scaler.data_min_[i]
    max_val = artifacts.scaler.data_max_[i]
    print(f"  {name:<28} {min_val:<15.2f} {max_val:.2f}")

Scaling ranges (learned from training data):
Feature                        Min (→0.0)      Max (→1.0)
------------------------------------------------------------
  tenure_months                1.00            60.00
  monthly_charges              29.85           105.45
  total_charges                29.85           6327.00
  support_tickets              0.00            5.00
  contract_type                0.00            2.00
  payment_method               0.00            2.00
  interaction_charges_x_tenure 29.85           6327.00


## Training Mode vs. Scoring Mode

This is the most critical concept in feature engineering:

- **Training mode** (`fit=True`): Look at the data, learn the rules ("median tenure is 29", "there are 3 contract types", "charges range 18-118"). Apply those rules.

- **Scoring mode** (`fit=False`): DON'T look at the new data to learn anything. Just apply the old rules from training.

**Why does this matter?** Imagine during training, tenure ranged 1-72. Months later, scoring data arrives with a customer at tenure=84 (longer than anyone in training). If we re-fit the scaler, `0.5` now means something different than what the model learned. The model would see a "different language" than it was trained on.

Scoring mode prevents this by ALWAYS using the training-time rules.

In [9]:
# New data arrives for scoring (different customers, different values)
scoring_data = pd.DataFrame({
    "customer_id": ["NEW_001", "NEW_002"],
    "tenure_months": [84, 6],           # 84 is beyond training range (1-60)!
    "monthly_charges": [120.0, 25.0],   # 120 is beyond training range!
    "total_charges": [10080.0, 150.0],
    "churn_label": [0, 1],
    "contract_type": ["two_year", "month-to-month"],
    "support_tickets": [0, 7],
    "payment_method": ["bank_transfer", "electronic_check"],
})

# Score using TRAINING-TIME artifacts (not re-fitting!)
scored_matrix, _ = engineer_features(scoring_data, fit=False, artifacts=artifacts)

print("Scoring new data with training-time rules:")
print(f"  Shape: {scored_matrix.shape}")
print(f"  Any NaN: {np.isnan(scored_matrix).any()}")
print(f"  Range: [{scored_matrix.min():.3f}, {scored_matrix.max():.3f}]")
print(f"\n  Notice: values are clipped to [0, 1] even though some exceeded")
print(f"  the training range. The model never sees out-of-range values.")

scored_df = pd.DataFrame(scored_matrix, columns=artifacts.feature_names)
print(f"\n{scored_df.round(3)}")

Scoring new data with training-time rules:
  Shape: (2, 7)
  Any NaN: False
  Range: [0.000, 1.000]

  Notice: values are clipped to [0, 1] even though some exceeded
  the training range. The model never sees out-of-range values.

   tenure_months  monthly_charges  total_charges  support_tickets  \
0          1.000              1.0          1.000              0.0   
1          0.085              0.0          0.019              1.0   

   contract_type  payment_method  interaction_charges_x_tenure  
0            1.0             0.0                         1.000  
1            0.0             1.0                         0.019  


## Idempotence: Scoring Mode is a Pure Function

A critical property: running scoring mode twice with the same artifacts produces **bit-for-bit identical** results. No randomness, no state changes. Same input + same artifacts = same output, always.

If this property breaks, your predictions become non-reproducible — a customer's risk score could change between two runs even though nothing about them changed.

In [10]:
# Run scoring mode twice — results must be identical
result_1, _ = engineer_features(scoring_data, fit=False, artifacts=artifacts)
result_2, _ = engineer_features(scoring_data, fit=False, artifacts=artifacts)

identical = np.array_equal(result_1, result_2)
print(f"Run 1 == Run 2: {identical}")
print(f"Max difference: {np.abs(result_1 - result_2).max()}")

if identical:
    print("\n✓ Scoring mode is idempotent — same input always gives same output")

Run 1 == Run 2: True
Max difference: 0.0

✓ Scoring mode is idempotent — same input always gives same output


## The Three Guarantees

No matter what valid data goes in, the feature engineering module guarantees:

1. **Same row count** — no customers disappear or duplicate
2. **All float64** — every cell is a number, ready for math
3. **Zero NaN values** — no blanks that would crash the model

We verify these with property-based testing (Hypothesis) — 200 random DataFrames, each with different sizes, null patterns, and column subsets.

In [11]:
# Quick demonstration with random data
print("Testing the three guarantees with random data:")
print("=" * 50)

for trial in range(5):
    n = np.random.randint(5, 100)
    random_df = pd.DataFrame({
        "customer_id": [f"R{i}" for i in range(n)],
        "tenure_months": np.random.choice([*np.random.randint(1, 72, n-2).tolist(), None, None]),
        "monthly_charges": np.random.uniform(18, 118, n).tolist(),
        "total_charges": np.random.uniform(100, 8000, n).tolist(),
        "churn_label": np.random.randint(0, 2, n).tolist(),
    })
    # Inject some random nulls
    for col in ["monthly_charges", "total_charges"]:
        mask = np.random.random(n) < 0.1
        random_df.loc[mask, col] = None
    
    matrix, _ = engineer_features(random_df, fit=True)
    
    row_check = matrix.shape[0] == n
    dtype_check = matrix.dtype == np.float64
    nan_check = not np.isnan(matrix).any()
    
    status = "✓" if all([row_check, dtype_check, nan_check]) else "✗"
    print(f"  Trial {trial+1}: {n} rows, nulls injected → "
          f"rows={row_check}, float64={dtype_check}, no_nan={nan_check} {status}")

print("\nAll guarantees hold!")

Testing the three guarantees with random data:
  Trial 1: 95 rows, nulls injected → rows=True, float64=True, no_nan=True ✓
  Trial 2: 56 rows, nulls injected → rows=True, float64=True, no_nan=True ✓
  Trial 3: 5 rows, nulls injected → rows=True, float64=True, no_nan=True ✓
  Trial 4: 93 rows, nulls injected → rows=True, float64=True, no_nan=True ✓
  Trial 5: 19 rows, nulls injected → rows=True, float64=True, no_nan=True ✓

All guarantees hold!


## The Artifacts: What Gets Saved

When training completes, the `FeatureArtifacts` object is saved to S3. It contains everything needed to reproduce the exact same transformation on future data:

- **Scaler** — the min/max values per feature
- **Encoders** — the category→number mapping per column
- **Impute values** — what to fill blanks with
- **Feature names** — the exact column order the model expects

Without these artifacts, scoring would be impossible — you'd have to retrain just to transform new data.

In [12]:
# Summary of what's stored in artifacts
print("FeatureArtifacts contents:")
print("=" * 50)
print(f"\n1. Scaler: MinMaxScaler fitted on {len(artifacts.feature_names)} features")
print(f"2. Encoders: {len(artifacts.encoders)} categorical columns encoded")
for col in artifacts.encoders:
    print(f"     - {col}: {len(artifacts.encoders[col].classes_)} categories")
print(f"3. Impute values: {len(artifacts.impute_values)} columns with fill values")
print(f"4. Feature names: {artifacts.feature_names}")
print(f"\nThis object is serialized (pickle) and stored alongside the model.")
print(f"Scoring jobs load it to ensure identical transformations.")

FeatureArtifacts contents:

1. Scaler: MinMaxScaler fitted on 7 features
2. Encoders: 2 categorical columns encoded
     - contract_type: 3 categories
     - payment_method: 3 categories
3. Impute values: 6 columns with fill values
4. Feature names: ['tenure_months', 'monthly_charges', 'total_charges', 'support_tickets', 'contract_type', 'payment_method', 'interaction_charges_x_tenure']

This object is serialized (pickle) and stored alongside the model.
Scoring jobs load it to ensure identical transformations.


## Key Takeaways

1. **Models only speak numbers** — feature engineering is the translation layer
2. **Four steps:** impute → encode → interact → scale
3. **Training mode learns rules, scoring mode applies them** — never re-learn during scoring
4. **Three guarantees:** same row count, all float64, zero NaN
5. **Artifacts are essential** — without them, you can't reproduce the transformation
6. **Idempotence in scoring mode** — same input always gives same output

Next: Chapter 4 shows how we feed this numeric matrix to XGBoost — a team of 100 simple decision-makers that learn from each other's mistakes.

---

## Run It For Real: Feature Engineer the IBM Telco Dataset

Chapter 2 saved `data/telco_validated.csv` — 7,043 validated customers. Now let's transform them into a numeric matrix the model can learn from.

**Input:** `data/telco_validated.csv` (from Chapter 2)  
**Output:** `data/telco_features.npy`, `data/telco_labels.npy`, `data/telco_artifacts.pkl`

In [ ]:
import os
import pickle

# Load the output from Chapter 2
DATA_DIR = "../data"
VALIDATED_PATH = os.path.join(DATA_DIR, "telco_validated.csv")

assert os.path.exists(VALIDATED_PATH), (
    f"File not found: {VALIDATED_PATH}\n"
    f"Run Chapter 2's 'Run It For Real' section first."
)

telco_validated = pd.read_csv(VALIDATED_PATH)
print(f"Loaded: {telco_validated.shape[0]} customers, {telco_validated.shape[1]} columns")

In [ ]:
# Run feature engineering in TRAINING mode (fit=True)
# This learns all transformation rules from the full dataset
features, artifacts = engineer_features(telco_validated, fit=True)
labels = telco_validated["churn_label"].values.astype(int)

print(f"Feature matrix: {features.shape}")
print(f"  → {features.shape[0]} customers × {features.shape[1]} numeric features")
print(f"  → dtype: {features.dtype}, NaN count: {np.isnan(features).sum()}")
print(f"  → value range: [{features.min():.3f}, {features.max():.3f}]")
print(f"\nLabels: {labels.shape[0]} values")
print(f"  → {(labels == 0).sum()} stayed (0), {(labels == 1).sum()} churned (1)")
print(f"  → churn rate: {labels.mean():.1%}")
print(f"\nFeatures used: {artifacts.feature_names}")

In [ ]:
# Save everything for Chapter 4
np.save(os.path.join(DATA_DIR, "telco_features.npy"), features)
np.save(os.path.join(DATA_DIR, "telco_labels.npy"), labels)
with open(os.path.join(DATA_DIR, "telco_artifacts.pkl"), "wb") as f:
    pickle.dump(artifacts, f)

print("Saved:")
print(f"  data/telco_features.npy  — {features.shape} numeric matrix")
print(f"  data/telco_labels.npy    — {labels.shape} binary labels")
print(f"  data/telco_artifacts.pkl — scaler, encoders, impute values, feature names")
print(f"\n  → Chapter 4 will load these and train an XGBoost model.")

---

*Source code: `src/churn_pipeline/steps/feature_engineering.py`*  
*Tests: `tests/property/test_feature_engineering.py`*  
*Series: [Build with AWS](https://buildwithaws.substack.com/)*